In [ ]:
import numpy as np
import tensorflow as tf
import gym
import matplotlib.pyplot as plt

### Returns Generator

In [ ]:
def generate_log_returns(mean, variance):
    return np.random.normal(mean, variance)
    # return 0.001

### Allocating assets

In [ ]:
def alloc_change(curr_cash_val, curr_stock_val, action):
    pv = curr_cash_val + curr_stock_val
    cash_percent = (curr_cash_val/pv)*100
    stock_percent = (curr_stock_val/pv)*100
   
    new_cash = (cash_percent - action)*0.01*pv
    new_stock = (stock_percent + action)*0.01*pv
    if new_cash>=0 and new_cash<=pv:
        return(new_cash, new_stock)
    else:
        return(curr_cash_val, curr_stock_val)

### Creating environment class.

In [ ]:
class env_asset_allocation:
    def __init__(self, initial_cash, initial_stock, start_price_change_dir, num_steps=1000):
        self.init_cash = initial_cash
        self.init_stock = initial_stock
        self.init_past_mov_dir = start_price_change_dir
        self.init_steps = num_steps
        self.cash = initial_cash
        self.stock = initial_stock
        self.past_mov_dir = start_price_change_dir
        self.steps_remaining = num_steps
        self.state_observation = np.array([self.cash, self.stock, self.past_mov_dir])
        self.done = False
        
    def get_actions(self):
        return np.array([-0.05, 0, 0.05])
        
    def check_is_done(self):
        return (self.steps_remaining == 0)
        
    def action(self, action_value):
        if self.check_is_done():
            self.done = True
            # raise Exception("Reached the end of the simulation")
        # Cash and Stocks after changing allocation based on the action.
        cash_new, stock_new = alloc_change(self.cash, self.stock, action_value)
        
        # Stock value after price movement and some log return.
        log_return = generate_log_returns(-0.001, 0.001)
        stock_new = stock_new * np.exp(log_return)
        
        # Portfolio value (PV) initial and final
        PV_initial = self.cash + self.stock
        PV_final = cash_new + stock_new
        
        # Reward as the final returns.
        reward = PV_final - PV_initial
        
        # State update after taking the action.
        self.past_mov_dir = np.sign(log_return)
        self.cash = cash_new
        self.stock = stock_new
        self.steps_remaining-=1
        self.state_observation = np.array([self.cash, self.stock, self.past_mov_dir])
        return self.state_observation, reward, self.done, self.steps_remaining
        
    def reset(self):
        self.cash = self.init_cash
        self.stock = self.init_stock
        self.past_mov_dir = self.init_past_mov_dir
        self.steps_remaining = self.init_steps
        self.state_observation = [self.cash, self.stock, self.past_mov_dir]
        self.done = False
        return self.state_observation

### Defining Q-Networks model architecture.

In [ ]:
# Define a simple neural network for Q-function approximation
class QNetwork(tf.keras.Model):
    def __init__(self, state_dim, action_dim):
        super(QNetwork, self).__init__()
        self.dense1 = tf.keras.layers.Dense(64, activation='relu', input_shape=(state_dim,))
        self.dense2 = tf.keras.layers.Dense(32, activation='relu')
        self.q_values = tf.keras.layers.Dense(action_dim)  # Output Q-values for each action
    
    def call(self, state):
        x = self.dense1(state)
        x = self.dense2(x)
        return self.q_values(x)

### Training

In [ ]:
# Define parameters for training
gamma = 0.9        # Discount factor for future rewards
epsilon = 0.1       # Exploration rate
learning_rate = 0.001
batch_size = 32
target_update_freq = 1000

# # Initialize environment and Q-network
# env = gym.make('CartPole-v1')
# state_dim = env.observation_space.shape[0]
# action_dim = env.action_space.n


env = env_asset_allocation(500, 500, -1, 1000)
state_dim = 1
action_dim = 3

action_dict = {0:[1,0,0], 1:[0,1,0], 2:[0,0,1]}

print(state_dim)
print(action_dim)

q_network = QNetwork(state_dim, action_dim)
target_network = QNetwork(state_dim, action_dim)
target_network.set_weights(q_network.get_weights())  # Initialize target network with the same weights

optimizer = tf.keras.optimizers.Adam(learning_rate)

# Epsilon-greedy policy for action selection
def epsilon_greedy(state, epsilon=0.1):
    if np.random.rand() < epsilon:
        return np.random.choice(action_dim)  # Random action (exploration)
    else:
        q_values = q_network(np.expand_dims(state, axis=0))  # Get Q-values from the network
        return np.argmax(q_values)  # Best action (exploitation)

# Compute the loss and gradients
def compute_loss(states, actions, rewards, next_states, done_flags):
    q_values = q_network(states)
    next_q_values = target_network(next_states)

    # Get the Q-values for the taken actions
    selected_q_values = tf.gather(q_values, actions, batch_dims=1)

    # Compute the target Q-value (Bellman equation)
    max_next_q_values = tf.reduce_max(next_q_values, axis=1)  # Max Q-value for next state
    target_q_values = rewards + gamma * (1 - done_flags) * max_next_q_values

    # Compute the loss
    loss = tf.reduce_mean(tf.square(selected_q_values - target_q_values))
    return loss

# Training loop
def train(env, q_network, target_network, optimizer, episodes=100):
    returns_list = []
    for episode in range(episodes):
        state = env.reset()
        done = False
        total_reward = 0

        # Store experience replay buffer
        states, actions, rewards, next_states, done_flags = [], [], [], [], []
        alloc_cash, alloc_stock = [], []
        total_rewards = []

        while not done:
            action = epsilon_greedy([state[-1]], epsilon=epsilon)
            (next_state, reward, done, _) = env.action([ -0.05, 0, 0.05 ][action])

            # Store experience
            states.append([state[-1]])
            actions.append(action)
            rewards.append(reward)
            next_states.append([next_state[-1]])
            done_flags.append(done)
            cash = next_state[0]
            stock = next_state[1]
            alloc_cash.append(cash/(cash+stock))
            alloc_stock.append(stock/(cash+stock))

            total_reward = reward + 0.9*total_reward
            state = next_state
            total_rewards.append(total_reward)

        # Convert to numpy arrays
        states = np.array(states)
        # print(states)
        actions = np.array(actions)
        # print(actions)
        rewards = np.array(rewards)
        next_states = np.array(next_states)
        done_flags = np.array(done_flags)
        
        returns_list.append(total_reward)
        
        plt.plot(alloc_cash, label='Cash')
        plt.plot(alloc_stock, label='Stock')
        plt.legend()
        plt.savefig(f'img_DQN_neg/{episode}.png')
        plt.show()
        
        plt.plot(total_rewards)
        plt.show()

        # Train on this batch of experiences
        with tf.GradientTape() as tape:
            loss = compute_loss(states, actions, rewards, next_states, done_flags)
        
        # Compute gradients and update the network
        gradients = tape.gradient(loss, q_network.trainable_variables)
        optimizer.apply_gradients(zip(gradients, q_network.trainable_variables))

        # Periodically update the target network
        # if episode % target_update_freq == 0:
        
        target_network.set_weights(q_network.get_weights())

        print(f'Episode {episode+1}, Total Reward: {total_reward}, Loss: {loss.numpy()}')
    return rewards, returns_list

In [ ]:
# Train the model
rewards, returns_list = train(env, q_network, target_network, optimizer, episodes=500)



In [ ]:
q_network(np.array([[-1]]))

In [ ]:
q_network(np.array([[0]]))

In [ ]:
q_network(np.array([[1]]))